In [0]:
%pylab inline

In [0]:
import dataiku
from dataiku import pandasutils as pdu
import pandas as pd

In [0]:
# Example: load a DSS dataset as a Pandas dataframe
mydataset = dataiku.Dataset("mydataset")
mydataset_df = mydataset.get_dataframe()

In [0]:
import dataiku
import pandas as pd

client = dataiku.api_client()
project = client.get_project("RECIPE_USAGE")
recipes = project.list_recipes()

data_summary = []

# Helper function to keep the code clean and avoid repeating logic
def get_bucket(recipe_type):
    if recipe_type == 'python':
        return "Data scientist group"
    elif recipe_type == 'sql_query':
        return "Data analyst group"
    else:
        return "Other / Visual Analyst"

for r_summary in recipes:
    recipe_name = r_summary['name']
    recipe_type = r_summary['type']
    
    recipe = project.get_recipe(recipe_name)
    settings = recipe.get_settings()
    raw_data = settings.data
    print(raw_data)
    # Access the nested recipe metadata
    recipe_meta = raw_data.get('recipe', {})
    
    # Extract Logins
    creator = recipe_meta.get('creationTag', {}).get('lastModifiedBy', {}).get('login', 'Unknown')
    last_modifier = recipe_meta.get('versionTag', {}).get('lastModifiedBy', {}).get('login', 'Unknown')
    
    # Assign Buckets
    # Both use the same logic based on the recipe type
    bucket_label = get_bucket(recipe_type)
    
    data_summary.append({
        "Recipe Name": recipe_name,
        "Type": recipe_type,
        "Created By": creator,
        "Created By Bucket": bucket_label,
        "Last Modified By": last_modifier,
        "Modified By Bucket": bucket_label
    })

# Create DataFrame
df = pd.DataFrame(data_summary)

# Display result
print(df.to_string(index=False))

# Optional: Write to Dataiku Dataset
# recipe_usage_ds = dataiku.Dataset("recipe_usage")
# recipe_usage_ds.write_with_schema(df)